![NCAR UCAR Logo](../NCAR_CISL_NSF_banner.jpeg)

#### Authors: Daniel Howard & Mrinal Biswas
HPCD Consulting Services Group, NSF NCAR
#### Date: May 26, 2026

# PBS Job Submissions for NSF NCAR HPC Systems

This notebook demonstrates how to:
- Create a PBS job script
- Submit a job to the scheduler
- Monitor job status
- Inspect job output

We will use shell commands and Python together to manage HPC workflows.

## Check Current Environment

Before submitting jobs, confirm your:
- Current directory
- Available files
- Loaded modules (if applicable)

Use the following code cell to do this. Reminder, execute code cells with `CTRL/CMD + ENTER` or copy the into a terminal session to run.

In [ ]:
%%bash
# List current directory
pwd

# List files in current directory
ls

# List loaded modules
ml # Short for module list

## Example: Batch script to run a hybrid MPI/OpenMP job on Derecho

### Step 1: Create Job Script

We will run a CPU program with MPI on Derecho. First, use the below cell to create a bash script file which we will submit to the PBS job scheduler.

Review the PBS preamble, ie the section that starts woth `#PBS`. Edit the line including `-A` with a new project code which you are authorized to use. If necessary, use the `sam-search` command cell to identify the projects you are a member of.

In [ ]:
!sam-search user $USER --list-projects

In [ ]:
%%writefile job_script.sh
#!/bin/bash
#PBS -A SCSG0001
#PBS -N HelloWorld
#PBS -q main@desched1
#PBS -l walltime=00:01:00
#PBS -l select=1:ncpus=128:mpiprocs=128
#PBS -o Hello_MPI.log

# Load modules to match compile-time environment
module --force purge
module load ncarenv/25.10 intel/2025.2.1 cray-mpich/8.1.32

# Build Hello World MPI program
mkdir -p bin
mpif90 src/Hello_MPI.f90 -o bin/Hello_MPI

# Run application with MPI binding helper script
mpibind ./bin/Hello_MPI


If you like, feel free to review the source file [Hello_MPI.f90](./src/Hello_MPI.f90) for the implementation of the parallel MPI Hello World example in Fortran.

### Step 2: Submit Job Script

The `qsub` command is the main way used to send jobs to the HPC system via the PBS job scheduler.

In order to submit our job created above, simply run the below cell. Upon completion, the command will print the assigned job ID.

In [ ]:
!qsub job_script.sh

You can check that status of your job with the `qstat` command or by reviewing your [Open OnDemand Active Jobs Dashboard](https://ondemand.hpc.ucar.edu/pun/sys/dashboard/activejobs).

- The flag `-x` is used to include recently completed jobs. 
- The flag `-n` requests that the assigned node(s) are included in the summary output.
- The flag `-u $USER` only displays jobs from the user which ran the command or can query other users. 
- The command `qstat -f [JOB_ID]` will query details on one specific job. Add `-f` to show verbose details.

In [ ]:
!qstat -x -n -u $USER

Print the job output. Run this only when the above job shows a `F` for finished state and the file `Hello_MPI.log`, to be written out per `#PBS -o Hello_MPI.log`, has been created.

In [ ]:
!cat Hello_MPI.log

### Question: Why does the ordering of MPI print statements show as out of order?

Expand to reveal an explanation.

The ordering of MPI print statements may appear out of order in this hello world example because each MPI process runs independently and may reach the print statement at different times. Timing jitters across a CPU and/or nodes, even if previously idle, won't guarentee ordered processing a scheduled parallel MPI tasks.

The output from each process is typically buffered, and the order in which the output is flushed to the terminal can vary based on factors such as the scheduling of processes, the buffering behavior of the output stream, and the way the MPI implementation handles output from multiple processes. As a result, the print statements from different processes may not appear in a sequential order, leading to an interleaved or seemingly out-of-order output.


#### Convenience Job Submission Scripts
Other convenience scripts are `qinteractive`, or `execcasper` for Casper, which will automatically launch the commands 

`qinteractive`: 
```bash
qsub -I -l select=1:ncpus=32:mem=55GB -A $PBS_ACCOUNT -q develop@desched1 -l walltime=01:00:00
```
or

`execcasper`:
```bash
qsub -I -l select=1:ncpus=1:mem=10GB -A $PBS_ACCOUNT -q casper@casper-pbs -l walltime=06:00:00
```

each creating an interactive job session on a compute node. Additional arguments may be used to refine the resource request such as `qinteractive -l select=1:ncpus=2 -A [PROJECT_ID] ...`.  If `$PBS_ACCOUNT` is not set, you must use the flag `-A [PROJECT_ID]` to specify an allocation to use.

You can set the `$PBS_ACCOUNT` environment variable with your preferred default project with the below command, edittting `[Project_ID]`. This will add the `export` command to your `~/.profile` file which is ran at the beginning of every user shell session.

In [ ]:
!echo "export PBS_ACCOUNT=[PROJECT_ID]" >> ~/.profile

Another convenience script is `qcmd` which is a non-interactive version of the previous commands (omits the `-I`) and allows to run a quickly referenced shell script or direct command such as `qcmd -- echo "Hello from qcmd!"`.

## Exercise: Try `qcmd`
Run the below cell and try modifying the number of CPUs or MPI processes requested. Keep in mind that `qcmd` executes commands or scripts but does not parse any `#PBS` preamble settings if passed a job script file. Also, output from `qcmd` will be sent directly back to the terminal or `STDOUT`.

If you like, try also re-running prior cells and modifying other job parameters in the job script, such as the `select` statement whcih requests a total number of nodes. 

How will these changes modify the output of `qcmd` below and `qsub` above?

In [ ]:
# Set the job_script.sh file to be executable and resubmit the job
!chmod +x job_script.sh
!qcmd -l select=1:ncpus=4:mpiprocs=4 -- ./job_script.sh